In [1]:
from fastai.vision.all import *
# from sklearn.metrics import roc_curve, auc
# from fastai.metrics import * 

# from pathlib import Path
from sklearn.model_selection import  train_test_split # GroupKFold, StratifiedGroupKFold, LeaveOneGroupOut, LeavePGroupsOut,
from sklearn.utils import resample
import sklearn.metrics as skm
from pathlib import Path
path = Path('/ix/rbao/Projects/HCC-CBS-068-Hillman-ASinghi-2/data/biliseq_he_class/proc/v5')
import numpy as np
from numpy import random
import shutil
import glob


In [2]:
# Create some regular expression queries:
(path/"tiles").ls()
fname = (path/"tiles").ls()[0]
print(fname.name)
re_class= r"class_(.+)_x\d+_y\d+.jpg"
out=re.findall(re_class,fname.name)
print(out)
re_slide =r'(.+)_class_\S+_x\d+_y\d+.jpg$'
out=re.findall(re_slide,fname.name)
print(out)
re_slide_class=r'(.+)_class_(.+)_x\d+_y\d+.jpg$'
slides=[]
slides_c=[]
all_c=[]
all_f=[]
for f in (path/"tiles").ls():
    all_f.append(f.name)
    slide, c = re.findall(re_slide_class,f.name)[0]
    slides_c.append("%s_%s" % (slide,c))
    slides.append(slide)
    all_c.append(c)
print(np.unique(slides_c), len(np.unique(slides_c)))

1007470_class_neg_x8000_y64500.jpg
['neg']
['1007470']
['1007466_neg' '1007467_neg' '1007468_neg' '1007469_neg' '1007470_neg'
 '1007471_neg' '1007473_neg' '1007474_neg' '1007476_pos' '1007477_pos'
 '1007478_pos' '1007482_pos' '1007484_pos' '1007485_pos' '1007486_pos'
 '1007720_neg' '1007726_neg' '1007731_pos' '1007733_pos' '1007820_neg'
 '1007821_neg' '1007822_neg' '1007824_neg' '1007825_neg' '1007826_pos'
 '1007827_pos' '1007828_pos' '1007829_pos' '1007830_pos' '1007831_pos'
 '1007832_pos' '1007845_neg' '1007846_neg' '1007847_neg' '1007848_neg'] 35


In [3]:
#Check that all .svs files actually result in tiles being extracted:
path_raw_pos = path.parent.parent.joinpath('raw').joinpath('bile_ducts_pos')
path_raw_neg = path.parent.parent.joinpath('raw').joinpath('bile_ducts_neg')
all_svs = path_raw_pos.ls() + path_raw_neg.ls()
# print(all_svs)
re_raw_class=r'(+.).svs$'
missing = 0
for raw in all_svs:
    r_id = raw.parts[-1].split('.')[0]
    if r_id not in slides:
        if 'pos' in str(raw):
            c='pos'
        else:
            c='neg'
        print('%s %s not found in processed tiles!' % (r_id,c))
        missing +=1
if missing == 0:
    print('All %d .svs files have processed tiles.' % (len(all_svs)))
        

All 35 .svs files have processed tiles.


In [4]:
def equal_class_by_slide(pos_set_groups,neg_set_groups,pos_set_fn,neg_set_fn,ntarget=175):
    all_orig=[pos_set_fn,neg_set_fn]
    nneg=[]
    npos=[]
    all_new = [np.empty((1,3)),np.empty((1,3))]
    #For each slide / group in pos/neg set_groups, limit tiles used to ntarget
    for i,tset in enumerate( [pos_set_groups, neg_set_groups]):
        set_idx=[]
        if i==0:
            print('\nPositive slides:')
            n=npos
        else:
            print('\nNegative slides:')
            n=nneg

        for ii,slide_id in enumerate(tset):
            idx = all_orig[i][:,2]==slide_id
            keep = all_orig[i][idx,:]
            if keep.shape[0] > ntarget:
                keep= resample(keep,
                      n_samples=ntarget,
                      random_state=42,
                      replace = False,)
            if ii==0:
                all_new[i]=keep
            else:
                all_new[i]=np.concatenate((all_new[i],keep),axis=0)
            ntiles=keep.shape[0]
            n.append(ntiles)
     
        print('\t n = %d svs' % len(tset))
        print('\t median n tiles / svs = %4.1f' % np.median(n))
    print('\nTraining set has %d pos and %d neg tiles after per-slide equalization' % (np.sum(npos),np.sum(nneg)))
    
    #Overall randomly resample the classes to be perfect match:
    min_rows = min(np.sum(npos), np.sum(nneg))
    print('Smaller of the 2: %d' % min_rows)
    res_pos =resample(all_new[0],
                      n_samples=min_rows,
                      random_state=42,
                      replace = False,)
    res_neg =resample(all_new[1],
                      n_samples=min_rows,
                      random_state=42,
                      replace = False,)
    new = [res_pos,res_neg]
    all_output=np.concatenate((res_pos,res_neg),axis=0)
    print('\nTraining set has %d pos and %d neg tiles after total equalization' % (res_pos.shape[0],res_neg.shape[0]))

    print('Total train slides after balance = %d' % all_output.shape[0])
    return all_output

In [5]:
# Create a limited /train /valid set that will be faster to train and will be nicely balanced etc:
#1) Pick 3 pos and 3 neg slides to randomly put in validation set
posf = np.array([(f,c,s) for f,c,s in zip(all_f,all_c,slides) if c =='pos'])
negf = np.array([(f,c,s) for f,c,s in zip(all_f,all_c,slides) if c =='neg'])
# print(negf[0])
print('Positive & negative tile n:',len(posf),len(negf))
ntarget = 175
seed=42 #Currently using 42
train_pos, valid_pos = train_test_split(np.unique(posf[:,2]),
                                       test_size=0.2,
                                       random_state=seed)
train_neg, valid_neg = train_test_split(np.unique(negf[:,2]),
                                       test_size=0.2,
                                       random_state=seed)
print('Using as validation set slides:')
print(valid_pos,valid_neg)

#2) For each set collect list of original filenames and balance by class:
print('\nTRAINING SET:')
all_train = equal_class_by_slide(train_pos, train_neg, posf, negf)

print('Total len:', len(all_train), 'Unique len:', len(np.unique(all_train[:,0])))

print('\nVALIDATION SET:')
all_valid = equal_class_by_slide(valid_pos, valid_neg, posf, negf)
print('Total len:', len(all_valid), 'Unique len:', len(np.unique(all_valid[:,0])))
print('Validation slides:', np.unique(all_valid[:,2]))

# ['1007476' '1007477' '1007485' '1007831'] ['1007466' '1007471' '1007821' '1007467']
# ['1007477' '1007486' '1007733' '1007826'] ['1007820' '1007467' '1007720' '1007848'] #0
# ['1007482' '1007830' '1007731' '1007478'] ['1007469' '1007845' '1007473' '1007820'] #1
# ['1007829' '1007484' '1007485' '1007476'] ['1007726' '1007470' '1007825' '1007466'] #2

Positive & negative tile n: 10145 5161
Using as validation set slides:
['1007476' '1007477' '1007485' '1007831'] ['1007466' '1007471' '1007821' '1007467']

TRAINING SET:

Positive slides:
	 n = 12 svs
	 median n tiles / svs = 175.0

Negative slides:
	 n = 15 svs
	 median n tiles / svs = 175.0

Training set has 2071 pos and 2224 neg tiles after per-slide equalization
Smaller of the 2: 2071

Training set has 2071 pos and 2071 neg tiles after total equalization
Total train slides after balance = 4142
Total len: 4142 Unique len: 4142

VALIDATION SET:

Positive slides:
	 n = 4 svs
	 median n tiles / svs = 175.0

Negative slides:
	 n = 4 svs
	 median n tiles / svs = 175.0

Training set has 600 pos and 554 neg tiles after per-slide equalization
Smaller of the 2: 554

Training set has 554 pos and 554 neg tiles after total equalization
Total train slides after balance = 1108
Total len: 1108 Unique len: 1108
Validation slides: ['1007466' '1007467' '1007471' '1007476' '1007477' '1007485' '1007821

In [183]:
#3) If happy with train and validation sets, move tiles over:

if path.joinpath('model').exists()==False:
    print('Creating model path: %s' % path.joinpath('model'))
    os.mkdir(str(path.joinpath('model')))

#Move train/valid into GrandParent structure: train/class test/class

#TRAIN:
src = path.joinpath('tiles')
dest = path.joinpath('model').joinpath('train')
print('Copy %d files from training set from %s to %s' % (all_train.shape[0],src,dest))
for f in all_train:
    fin_dest=dest.joinpath(f[1])
    if fin_dest.exists()==False:
        print('Creating model path: %s' % fin_dest)
        os.makedirs(str(fin_dest))
    if fin_dest.joinpath(f[0]).exists()==False:
        shutil.copyfile(src.joinpath(f[0]),fin_dest.joinpath(f[0]))
ftest=glob.glob(str(dest.joinpath('**/*.jpg')),recursive=True)
print('%d files found in %s' % (len(ftest),dest))

#VALID
dest =  path.joinpath('model').joinpath('valid')
print('Copy %d files from training set from %s to %s' % (all_valid.shape[0],src,dest))
for f in all_valid:
    fin_dest=dest.joinpath(f[1])
    if fin_dest.exists()==False:
        print('Creating model path: %s' % fin_dest)
        os.makedirs(str(fin_dest))
    if fin_dest.joinpath(f[0]).exists()==False:
        shutil.copyfile(src.joinpath(f[0]),fin_dest.joinpath(f[0]))
ftest=glob.glob(str(dest.joinpath('**/*.jpg')),recursive=True)
print('%d files found in %s' % (len(ftest),dest))
print('Finished!')

Copy 4142 files from training set from /ix/rbao/Projects/HCC-CBS-068-Hillman-ASinghi-2/data/biliseq_he_class/proc/v5/tiles to /ix/rbao/Projects/HCC-CBS-068-Hillman-ASinghi-2/data/biliseq_he_class/proc/v5/model/train
Creating model path: /ix/rbao/Projects/HCC-CBS-068-Hillman-ASinghi-2/data/biliseq_he_class/proc/v5/model/train/pos
Creating model path: /ix/rbao/Projects/HCC-CBS-068-Hillman-ASinghi-2/data/biliseq_he_class/proc/v5/model/train/neg
4142 files found in /ix/rbao/Projects/HCC-CBS-068-Hillman-ASinghi-2/data/biliseq_he_class/proc/v5/model/train
Copy 1108 files from training set from /ix/rbao/Projects/HCC-CBS-068-Hillman-ASinghi-2/data/biliseq_he_class/proc/v5/tiles to /ix/rbao/Projects/HCC-CBS-068-Hillman-ASinghi-2/data/biliseq_he_class/proc/v5/model/valid
Creating model path: /ix/rbao/Projects/HCC-CBS-068-Hillman-ASinghi-2/data/biliseq_he_class/proc/v5/model/valid/pos
Creating model path: /ix/rbao/Projects/HCC-CBS-068-Hillman-ASinghi-2/data/biliseq_he_class/proc/v5/model/valid/ne

In [15]:
#TEST:
#Use 'full' to create unbalanced test set that has full-slide worth of images:
method = 'full'
dest = path.joinpath('model').joinpath('test')
#valid = path.joinpath('model').joinpath('valid')
src = path.joinpath('tiles')
valid_slides = list(valid_pos) + list(valid_neg)
all_tiles=np.concatenate((posf,negf),axis=0)
for s in valid_slides:
    fns=[fcs for fcs in all_tiles if s in fcs[2]]
    # print('%d tiles found for slide %s' % (len(fns),s))
    for f in fns:
        if (f[0] not in all_valid[:,0]) or (method=='full'):
            c = f[1]
            fin_dest = dest.joinpath(c)
            if fin_dest.exists()==False:
                print('Creating model path: %s' % fin_dest)
                os.makedirs(str(fin_dest))
            if fin_dest.joinpath(f[0]).exists()==False:
                # print(fin_dest.joinpath(f[0]))
                shutil.copyfile(src.joinpath(f[0]),fin_dest.joinpath(f[0]))
ftest=glob.glob(str(dest.joinpath('**/*.jpg')),recursive=True)
print('%d files found in %s' % (len(ftest),dest))

Creating model path: /ix/rbao/Projects/HCC-CBS-068-Hillman-ASinghi-2/data/biliseq_he_class/proc/v5/model/test/pos
Creating model path: /ix/rbao/Projects/HCC-CBS-068-Hillman-ASinghi-2/data/biliseq_he_class/proc/v5/model/test/neg
2847 files found in /ix/rbao/Projects/HCC-CBS-068-Hillman-ASinghi-2/data/biliseq_he_class/proc/v5/model/test
